# Hierarchical Quaternion BTC 4-Hourly Experiment

Colab notebook for running the new LunarCrush hierarchical quaternion experiment.

This notebook assumes you want to run the BTC 4-hourly `16 -> 4 -> 1` hierarchy using:
- `configs/data/4hourly/btc_lunar_hierarchy.yaml`
- `configs/experiments/4hourly_hierarchical_quaternion.yaml`

Important: this experiment expects the cached dataset file `data/cache/lunarcrush_btc_4hour_full.csv` to exist inside the repo.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

IN_COLAB = 'google.colab' in sys.modules
REPO_DIR = Path('/content/thesis') if IN_COLAB else Path.cwd()
REPO_URL = 'https://github.com/YOUR_USERNAME/thesis.git'

print('IN_COLAB =', IN_COLAB)
print('REPO_DIR =', REPO_DIR)
print('REPO_URL =', REPO_URL)

if IN_COLAB and not REPO_DIR.exists():
    if 'YOUR_USERNAME' in REPO_URL:
        raise ValueError('Set REPO_URL to your GitHub repository before running this cell.')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())

In [ ]:
import subprocess
import sys

packages = [
    'torch',
    'pyyaml',
    'pandas',
    'numpy',
    'scipy',
    'matplotlib',
    'seaborn',
    'yfinance',
    'scikit-learn',
    'statsmodels',
    'tqdm',
]

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages, check=True)
print('Dependencies installed.')

In [ ]:
from pathlib import Path
import torch

data_file = Path('data/cache/lunarcrush_btc_4hour_full.csv')
if not data_file.exists():
    raise FileNotFoundError(
        'Missing data/cache/lunarcrush_btc_4hour_full.csv. Upload or sync the cached LunarCrush BTC 4-hour file into the repo before training.'
    )

print('CUDA available:', torch.cuda.is_available())
print('MPS available:', hasattr(torch.backends, 'mps') and torch.backends.mps.is_available())
print('Dataset file:', data_file.resolve())

In [ ]:
import yaml
from pathlib import Path

BASE_CONFIG = Path('configs/data/4hourly/btc_lunar_hierarchy.yaml')
EXPERIMENT_CONFIG = Path('configs/experiments/4hourly_hierarchical_quaternion.yaml')
RUN_MODE = 'smoke'  # change to 'full' for the full seeded thesis run

with BASE_CONFIG.open() as f:
    base_cfg = yaml.safe_load(f)

with EXPERIMENT_CONFIG.open() as f:
    exp_cfg = yaml.safe_load(f)

if torch.cuda.is_available():
    base_cfg['device'] = 'cuda'
else:
    base_cfg['device'] = 'cpu'

tmp_dir = Path('tmp_colab_configs')
tmp_dir.mkdir(exist_ok=True)
runtime_base = tmp_dir / 'btc_lunar_hierarchy_runtime.yaml'
runtime_exp = tmp_dir / '4hourly_hierarchical_quaternion_runtime.yaml'

if RUN_MODE == 'smoke':
    base_cfg['training']['num_epochs'] = 3
    base_cfg['training']['patience'] = 2
    exp_cfg['experiment']['seeds'] = [42]

with runtime_base.open('w') as f:
    yaml.safe_dump(base_cfg, f, sort_keys=False)

with runtime_exp.open('w') as f:
    yaml.safe_dump(exp_cfg, f, sort_keys=False)

print('RUN_MODE =', RUN_MODE)
print('Runtime base config:', runtime_base)
print('Runtime experiment config:', runtime_exp)

In [ ]:
import torch
from src.models.hierarchical_qnn import (
    HierarchicalQNNAttentionModel,
    HierarchicalQuaternionLSTMNoAttention,
)

x = torch.randn(2, 30, 16)
for cls in (HierarchicalQuaternionLSTMNoAttention, HierarchicalQNNAttentionModel):
    model = cls(hidden_size=64, num_layers=1, dropout=0.1, input_size=16, num_features=16, target_col=3, seq_len=30)
    y = model(x)
    print(cls.__name__, tuple(y.shape))

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, 'experiments/run_experiments.py',
    '--base-config', 'tmp_colab_configs/btc_lunar_hierarchy_runtime.yaml',
    '--experiment-config', 'tmp_colab_configs/4hourly_hierarchical_quaternion_runtime.yaml',
]

print('Running command:')
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json
from pathlib import Path

results_dir = Path('experiments/results/4hourly_btc_lunar_hierarchy')
result_files = sorted(results_dir.glob('*.json'))
if not result_files:
    raise FileNotFoundError(f'No result files found in {results_dir}')

latest = result_files[-1]
print('Latest result file:', latest)

with latest.open() as f:
    results = json.load(f)

table = []
for model_name, model_data in results['model_results'].items():
    agg = model_data['aggregated']
    table.append({
        'model': model_name,
        'mape_mean': agg['mape']['mean'],
        'dir_acc_mean': agg['directional_accuracy']['mean'],
        'dir_acc_3c_mean': agg['directional_accuracy_3class']['mean'],
        'sharpe_mean': agg['sharpe_ratio']['mean'],
        'sharpe_3c_mean': agg['sharpe_ratio_3class']['mean'],
    })

table